# pChEMBL Prediction: ML Ensemble Training

**Training ML models to predict binding affinity (pChEMBL) on molecules WITH bioactivity data**

**Training Dataset:** 42,113 molecules with experimental bioactivity (21% of full dataset)
**Split:** 70% train (29,479), 20% validation (8,423), 10% test (4,211)

**Models:**
- Random Forest Regressor (200 trees, max_depth=20)
- Gradient Boosting Regressor (200 trees, lr=0.05)
- Ensemble: Average of both predictions

**Expected Performance:** MAE ~0.5-0.8, R² ~0.5-0.7

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ Libraries imported successfully")

## 2. Load Data and Split by Bioactivity

In [ ]:
# Load dataset
data_path = Path("../chembl_curated_out/molecules_200k_with_bioactivity_21pct_coverage.csv")
df = pd.read_csv(data_path)

print(f"✅ Dataset loaded: {len(df):,} molecules")
print(f"   Total features: {len(df.columns)} columns")

# Extract ONLY molecules WITH bioactivity (42,113 molecules)
df_with_bio = df[df['has_bioactivity'] == True].copy()

print(f"\n📊 Training dataset:")
print(f"   WITH bioactivity: {len(df_with_bio):,} molecules (21.1% of full dataset)")
print(f"   Target variable: pchembl_mean (binding affinity)")
print(f"   Range: {df_with_bio['pchembl_mean'].min():.2f} - {df_with_bio['pchembl_mean'].max():.2f}")
print(f"   Mean: {df_with_bio['pchembl_mean'].mean():.2f} ± {df_with_bio['pchembl_mean'].std():.2f}")

# Save the 42,113 molecules to a separate CSV
output_file = Path("../chembl_curated_out/molecules_42k_with_bioactivity_training.csv")
df_with_bio.to_csv(output_file, index=False)
print(f"\n✅ Training dataset saved: {output_file}")
print(f"   Size: {df_with_bio.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

## 3. Feature Selection and Preprocessing

In [ ]:
# Exclude metadata and target columns
exclude_cols = [
    'molecule_chembl_id', 'canonical_smiles', 'first_approval', 'max_phase', 
    'pref_name', 'bemis_murcko_scaffold',  # Metadata
    'pchembl_mean', 'pchembl_median', 'pchembl_std', 'pchembl_min', 'pchembl_max',  # Target variables
    'activity_count', 'target_count', 'assay_count', 'target_names',  # Bioactivity metadata
    'has_bioactivity', 'is_active', 'is_potent', 'is_highly_potent'  # Flags
]

feature_cols = [col for col in df.columns if col not in exclude_cols]

print(f"✅ Feature selection:")
print(f"   Selected features: {len(feature_cols)}")
print(f"   Excluded columns: {len(exclude_cols)}")
print(f"\n   First 10 features: {feature_cols[:10]}")

# Prepare X and y for training (molecules WITH bioactivity only)
X_full = df_with_bio[feature_cols].values
y_full = df_with_bio['pchembl_mean'].values

print(f"\n📐 Training data shape:")
print(f"   Features (X): {X_full.shape}")
print(f"   Target (y): {y_full.shape}")

## 4. Train/Val/Test Split (70/20/10)

In [ ]:
# Split: 70% train, 20% val, 10% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_full, y_full, test_size=0.10, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.222, random_state=42  # 0.222 ≈ 20% of 90%
)

print(f"✅ Data split:")
print(f"   Train: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X_full)*100:.1f}%)")
print(f"   Val:   {X_val.shape[0]:,} samples ({X_val.shape[0]/len(X_full)*100:.1f}%)")
print(f"   Test:  {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X_full)*100:.1f}%)")

# Feature scaling (optional for tree models, but kept for consistency)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ Features scaled (StandardScaler): mean=0, std=1")
print(f"   Note: Tree models don't require scaling, but it doesn't hurt")

## 5. Train Random Forest Model

In [ ]:
%%time
# Train Random Forest (optimized for noisy bioactivity data)
rf_model = RandomForestRegressor(
    n_estimators=400,
    max_depth=16,              # Reduced from 20 to prevent overfitting
    min_samples_split=5,
    min_samples_leaf=5,        # Increased from 2 to reduce spurious splits
    max_features='sqrt',       # ~10 features per split (good for correlated descriptors)
    n_jobs=-1,
    random_state=42,
    verbose=0
)

rf_model.fit(X_train_scaled, y_train)

# Predictions
y_val_pred_rf = rf_model.predict(X_val_scaled)

# Metrics
mae_rf = mean_absolute_error(y_val, y_val_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_val, y_val_pred_rf))
r2_rf = r2_score(y_val, y_val_pred_rf)

print(f"\n🌲 Random Forest - Validation Metrics:")
print(f"   MAE:  {mae_rf:.3f} pChEMBL units")
print(f"   RMSE: {rmse_rf:.3f}")
print(f"   R²:   {r2_rf:.3f}")

## 6. Train Gradient Boosting Model

In [ ]:
%%time
# Train Gradient Boosting (optimized for noisy bioactivity data)
gb_model = GradientBoostingRegressor(
    n_estimators=700,          # Increased from 200 for better convergence
    max_depth=3,               # Reduced from 7 (critical for GB generalization)
    learning_rate=0.03,        # Reduced from 0.05 to match increased n_estimators
    subsample=0.8,
    min_samples_split=5,
    min_samples_leaf=5,        # Increased from 2 to prevent overfitting
    random_state=42,
    verbose=0
)

gb_model.fit(X_train_scaled, y_train)

# Predictions
y_val_pred_gb = gb_model.predict(X_val_scaled)

# Metrics
mae_gb = mean_absolute_error(y_val, y_val_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_val, y_val_pred_gb))
r2_gb = r2_score(y_val, y_val_pred_gb)

print(f"\n🚀 Gradient Boosting - Validation Metrics:")
print(f"   MAE:  {mae_gb:.3f} pChEMBL units")
print(f"   RMSE: {rmse_gb:.3f}")
print(f"   R²:   {r2_gb:.3f}")

## 7. Ensemble Model and Test Set Evaluation

In [ ]:
# Ensemble: Average predictions from both models
y_val_pred_ensemble = (y_val_pred_rf + y_val_pred_gb) / 2
y_test_pred_rf = rf_model.predict(X_test_scaled)
y_test_pred_gb = gb_model.predict(X_test_scaled)
y_test_pred_ensemble = (y_test_pred_rf + y_test_pred_gb) / 2

# Validation metrics
mae_val_ensemble = mean_absolute_error(y_val, y_val_pred_ensemble)
rmse_val_ensemble = np.sqrt(mean_squared_error(y_val, y_val_pred_ensemble))
r2_val_ensemble = r2_score(y_val, y_val_pred_ensemble)

# Test metrics
mae_test = mean_absolute_error(y_test, y_test_pred_ensemble)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred_ensemble))
r2_test = r2_score(y_test, y_test_pred_ensemble)

print(f"🎯 ENSEMBLE (RF + GB Average) - Final Performance:")
print(f"\n   Validation Set:")
print(f"      MAE:  {mae_val_ensemble:.3f} pChEMBL units")
print(f"      RMSE: {rmse_val_ensemble:.3f}")
print(f"      R²:   {r2_val_ensemble:.3f}")
print(f"\n   Test Set:")
print(f"      MAE:  {mae_test:.3f} pChEMBL units")
print(f"      RMSE: {rmse_test:.3f}")
print(f"      R²:   {r2_test:.3f}")

## 8. Feature Importance Analysis

In [ ]:
# Get feature importance from Random Forest
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

# Plot top 20 features
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(20)
plt.barh(range(len(top_features)), top_features['importance'], color='steelblue')
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance Score')
plt.title('Top 20 Most Important Features for pChEMBL Prediction')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("📊 Top 10 Features:")
for idx, row in feature_importance.head(10).iterrows():
    print(f"   {row['feature']:<30} {row['importance']:.4f}")

## 9. Save Trained Models

In [ ]:
# Save models
output_dir = Path("../models/pchembl_predictor")
output_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(rf_model, output_dir / 'random_forest_model.pkl')
joblib.dump(gb_model, output_dir / 'gradient_boosting_model.pkl')
joblib.dump(scaler, output_dir / 'feature_scaler.pkl')

print(f"✅ Models saved to {output_dir}/")
print(f"   - random_forest_model.pkl")
print(f"   - gradient_boosting_model.pkl")
print(f"   - feature_scaler.pkl")

print(f"\n🎉 Training complete!")
print(f"   Trained on: {len(X_train):,} molecules")
print(f"   Validated on: {len(X_val):,} molecules")
print(f"   Tested on: {len(X_test):,} molecules")
print(f"   Final Test R²: {r2_test:.3f}")
print(f"   Final Test MAE: {mae_test:.3f} pChEMBL units")